In [ ]:
!mkdir -p project/agents project/tools project/memory project/core
!touch project/__init__.py project/agents/__init__.py project/tools/__init__.py project/memory/__init__.py project/core/__init__.py


In [ ]:
%%writefile project/tools/tools.py
from typing import Dict, List, Any


def calculator(expression: str) -> str:
    """Safe simple calculator for basic arithmetic only."""
    allowed = set("0123456789+-*/(). %")
    if not expression or any(ch not in allowed for ch in expression):
        return "Invalid expression."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Calculation error: {exc}"


def summarizer(text: str, max_sentences: int = 3) -> str:
    """Simple extractive summarizer."""
    if not text:
        return ""
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    return ". ".join(sentences[:max_sentences]) + ("." if sentences else "")


def search(query: str) -> Dict[str, Any]:
    """Offline trusted-source style search mock for Colab demo."""
    q = query.lower()
    knowledge = {
        "fever": "Fever can happen due to infections or inflammation. Drink fluids, rest, and monitor temperature. Seek medical care if fever is high, persistent, or with breathing difficulty, confusion, severe weakness, chest pain, dehydration, or rash.",
        "cough": "Cough may occur with common cold, allergies, asthma, flu, or other infections. Seek care if cough has blood, severe breathlessness, chest pain, or lasts more than 2-3 weeks.",
        "dehydration": "Warning signs of dehydration include extreme thirst, very little urine, dizziness, dry mouth, sunken eyes, confusion, and weakness. Severe dehydration needs urgent medical care.",
        "dengue": "Dengue prevention includes removing stagnant water, using mosquito nets, wearing long sleeves, and using repellents. Warning signs include severe abdominal pain, persistent vomiting, bleeding, breathing difficulty, or extreme weakness.",
        "first aid": "For basic first aid, keep the person safe, check breathing, stop bleeding with pressure, cool burns with running water, and seek emergency help for severe injury.",
        "emergency": "Emergency warning signs include severe breathing difficulty, chest pain, unconsciousness, severe bleeding, stroke-like symptoms, seizures, severe allergic reaction, or confusion."
    }
    results = []
    for key, value in knowledge.items():
        if key in q:
            results.append({"source": "Trusted public-health demo knowledge base", "content": value})
    if not results:
        results.append({
            "source": "Trusted public-health demo knowledge base",
            "content": "For health concerns, use trusted public health sources and consult a qualified healthcare professional. This assistant gives general awareness only, not diagnosis."
        })
    return {"query": query, "results": results}


def red_flag_checker(user_input: str) -> Dict[str, Any]:
    """Detects urgent medical red flags from user text."""
    text = user_input.lower()
    red_flags = [
        "chest pain", "can't breathe", "cannot breathe", "difficulty breathing",
        "unconscious", "fainting", "severe bleeding", "blood in cough",
        "confusion", "seizure", "stroke", "blue lips", "severe dehydration",
        "persistent vomiting", "suicidal", "poison", "overdose"
    ]
    found = [flag for flag in red_flags if flag in text]
    return {
        "has_red_flags": bool(found),
        "red_flags": found,
        "recommendation": "Seek urgent medical help immediately." if found else "No urgent red-flag phrase detected from the message."
    }


def health_resource_finder(location: str = "") -> Dict[str, Any]:
    """Demo resource finder. Replace with Maps/API in deployment."""
    if not location:
        location = "your local area"
    return {
        "location": location,
        "resources": [
            "Nearest primary health center or clinic",
            "Government hospital emergency department",
            "Local ambulance/emergency helpline",
            "Accredited community health worker or pharmacist for basic guidance"
        ]
    }


In [ ]:
%%writefile project/memory/session_memory.py
from typing import Dict, List, Any


class SessionMemory:
    def __init__(self):
        self.messages: List[Dict[str, str]] = []
        self.profile: Dict[str, Any] = {
            "language": "English",
            "location": None,
            "preferences": {}
        }

    def add_message(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})

    def get_recent_messages(self, limit: int = 6) -> List[Dict[str, str]]:
        return self.messages[-limit:]

    def set_preference(self, key: str, value: Any) -> None:
        self.profile["preferences"][key] = value

    def set_location(self, location: str) -> None:
        self.profile["location"] = location

    def get_context(self) -> Dict[str, Any]:
        return {
            "recent_messages": self.get_recent_messages(),
            "profile": self.profile
        }


In [ ]:
%%writefile project/core/observability.py
from datetime import datetime
from typing import Dict, Any, List


class ObservabilityLogger:
    def __init__(self):
        self.logs: List[Dict[str, Any]] = []

    def log(self, event: str, data: Dict[str, Any] | None = None) -> None:
        self.logs.append({
            "timestamp": datetime.utcnow().isoformat(),
            "event": event,
            "data": data or {}
        })

    def get_logs(self) -> List[Dict[str, Any]]:
        return self.logs

    def print_logs(self) -> None:
        for log in self.logs:
            print(f"[{log['timestamp']}] {log['event']}: {log['data']}")


In [ ]:
%%writefile project/core/a2a_protocol.py
from dataclasses import dataclass, asdict
from typing import Dict, Any


@dataclass
class A2AMessage:
    sender: str
    receiver: str
    task: str
    context: Dict[str, Any]
    safety_level: str = "medical_safe"

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def create_message(sender: str, receiver: str, task: str, context: Dict[str, Any], safety_level: str = "medical_safe") -> Dict[str, Any]:
    return A2AMessage(
        sender=sender,
        receiver=receiver,
        task=task,
        context=context,
        safety_level=safety_level
    ).to_dict()


In [ ]:
%%writefile project/core/context_engineering.py
from typing import Dict, Any


SAFETY_RULES = [
    "Do not diagnose.",
    "Do not prescribe medicines.",
    "Give general health awareness only.",
    "Mention red flags clearly.",
    "Recommend a qualified healthcare professional when symptoms are serious, persistent, or unclear.",
    "For emergency red flags, advise urgent medical help immediately."
]


def build_context(user_input: str, memory_context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "user_input": user_input,
        "memory": memory_context,
        "safety_rules": SAFETY_RULES,
        "response_style": {
            "tone": "simple, calm, helpful",
            "format": "short sections with bullets",
            "disclaimer": "This is general health information, not medical diagnosis."
        },
        "trusted_source_policy": "Use trusted public-health style information and avoid unsupported claims."
    }


In [ ]:
%%writefile project/agents/planner.py
from typing import Dict, Any
from project.core.a2a_protocol import create_message


class PlannerAgent:
    def classify_task(self, user_input: str) -> str:
        text = user_input.lower()
        if any(word in text for word in ["emergency", "chest pain", "can't breathe", "unconscious", "severe bleeding"]):
            return "emergency_guidance"
        if any(word in text for word in ["near me", "hospital", "clinic", "resource", "doctor"]):
            return "resource_lookup"
        if any(word in text for word in ["prevent", "prevention", "dengue", "mosquito", "vaccine"]):
            return "preventive_health"
        if any(word in text for word in ["fever", "cough", "pain", "headache", "vomit", "dehydration", "symptom"]):
            return "symptom_guidance"
        return "general_health_awareness"

    def select_tools(self, task_type: str) -> list[str]:
        tool_map = {
            "emergency_guidance": ["red_flag_checker", "summarizer"],
            "resource_lookup": ["health_resource_finder", "summarizer"],
            "preventive_health": ["search", "summarizer", "red_flag_checker"],
            "symptom_guidance": ["search", "summarizer", "red_flag_checker"],
            "general_health_awareness": ["search", "summarizer"]
        }
        return tool_map.get(task_type, ["search", "summarizer"])

    def plan(self, context: Dict[str, Any]) -> Dict[str, Any]:
        user_input = context["user_input"]
        task_type = self.classify_task(user_input)
        tools = self.select_tools(task_type)

        plan_context = {
            "task_type": task_type,
            "tools": tools,
            "original_context": context,
            "instructions": [
                "Prepare safe, non-diagnostic health guidance.",
                "Use simple language.",
                "Include red flags where relevant.",
                "Recommend professional medical care when needed."
            ]
        }

        return create_message(
            sender="PlannerAgent",
            receiver="WorkerAgent",
            task="execute_health_guidance_plan",
            context=plan_context,
            safety_level="medical_safe"
        )


planner = PlannerAgent()


In [ ]:
%%writefile project/agents/worker.py
from typing import Dict, Any
from project.tools.tools import search, summarizer, calculator, red_flag_checker, health_resource_finder
from project.core.a2a_protocol import create_message


class WorkerAgent:
    def execute(self, planner_message: Dict[str, Any]) -> Dict[str, Any]:
        ctx = planner_message["context"]
        original_context = ctx["original_context"]
        user_input = original_context["user_input"]
        task_type = ctx["task_type"]
        tools = ctx["tools"]

        red_flags = red_flag_checker(user_input) if "red_flag_checker" in tools else {"has_red_flags": False, "red_flags": []}

        search_result = None
        if "search" in tools:
            search_result = search(user_input)

        resource_result = None
        if "health_resource_finder" in tools:
            location = original_context.get("memory", {}).get("profile", {}).get("location") or ""
            resource_result = health_resource_finder(location)

        draft = self._compose_response(
            user_input=user_input,
            task_type=task_type,
            red_flags=red_flags,
            search_result=search_result,
            resource_result=resource_result
        )

        draft = summarizer(draft, max_sentences=12)

        return create_message(
            sender="WorkerAgent",
            receiver="EvaluatorAgent",
            task="evaluate_health_response",
            context={
                "draft_response": draft,
                "task_type": task_type,
                "red_flags": red_flags,
                "used_tools": tools,
                "user_input": user_input
            },
            safety_level="medical_safe"
        )

    def _compose_response(self, user_input: str, task_type: str, red_flags: Dict[str, Any], search_result: Dict[str, Any] | None, resource_result: Dict[str, Any] | None) -> str:
        lines = []

        lines.append("This is general health information, not a medical diagnosis.")

        if red_flags.get("has_red_flags"):
            lines.append("Urgent warning: Your message includes possible red-flag symptoms: " + ", ".join(red_flags.get("red_flags", [])) + ".")
            lines.append("Please seek urgent medical help or contact local emergency services immediately.")
            return " ".join(lines)

        if task_type == "resource_lookup" and resource_result:
            lines.append(f"Possible health resources for {resource_result['location']} include: " + "; ".join(resource_result["resources"]) + ".")
            lines.append("For serious or worsening symptoms, visit a qualified healthcare professional.")
            return " ".join(lines)

        if search_result:
            contents = [item["content"] for item in search_result.get("results", [])]
            lines.append("Helpful information: " + " ".join(contents))

        lines.append("General safe steps: rest, drink fluids if appropriate, monitor symptoms, and avoid self-medicating without professional advice.")
        lines.append("Seek medical care if symptoms are severe, worsening, persistent, or if you feel unsafe.")

        return " ".join(lines)


worker = WorkerAgent()


In [ ]:
%%writefile project/agents/evaluator.py
from typing import Dict, Any


class EvaluatorAgent:
    def evaluate(self, worker_message: Dict[str, Any]) -> Dict[str, Any]:
        ctx = worker_message["context"]
        draft = ctx["draft_response"]

        unsafe_terms = [
            "you definitely have",
            "confirmed diagnosis",
            "take antibiotic",
            "take antibiotics",
            "ignore",
            "no need to see a doctor"
        ]

        issues = []
        lower = draft.lower()

        for term in unsafe_terms:
            if term in lower:
                issues.append(f"Unsafe phrase detected: {term}")

        if "not a medical diagnosis" not in lower:
            issues.append("Missing medical disclaimer.")

        if ctx.get("red_flags", {}).get("has_red_flags") and "urgent" not in lower:
            issues.append("Red flag detected but urgent advice missing.")

        approved = len(issues) == 0

        if not approved:
            final_response = (
                "This is general health information, not a medical diagnosis. "
                "Your question may need professional medical review. "
                "Please consult a qualified healthcare professional, and seek urgent help if symptoms are severe or worsening."
            )
        else:
            final_response = draft

        return {
            "approved": approved,
            "issues": issues,
            "response": final_response,
            "metadata": {
                "task_type": ctx.get("task_type"),
                "used_tools": ctx.get("used_tools", [])
            }
        }


evaluator = EvaluatorAgent()


In [ ]:
%%writefile project/main_agent.py
from typing import Dict, Any
from project.agents.planner import planner
from project.agents.worker import worker
from project.agents.evaluator import evaluator
from project.memory.session_memory import SessionMemory
from project.core.context_engineering import build_context
from project.core.observability import ObservabilityLogger


class MainAgent:
    def __init__(self):
        self.memory = SessionMemory()
        self.logger = ObservabilityLogger()

    def handle_message(self, user_input: str) -> Dict[str, Any]:
        self.logger.log("user_message_received", {"user_input": user_input})
        self.memory.add_message("user", user_input)

        memory_context = self.memory.get_context()
        context = build_context(user_input, memory_context)
        self.logger.log("context_built", {"context_keys": list(context.keys())})

        planner_message = planner.plan(context)
        self.logger.log("planner_completed", planner_message)

        worker_message = worker.execute(planner_message)
        self.logger.log("worker_completed", {
            "task": worker_message.get("task"),
            "used_tools": worker_message.get("context", {}).get("used_tools", [])
        })

        evaluation = evaluator.evaluate(worker_message)
        self.logger.log("evaluator_completed", {
            "approved": evaluation["approved"],
            "issues": evaluation["issues"]
        })

        final_response = evaluation["response"]
        self.memory.add_message("assistant", final_response)

        return {
            "response": final_response,
            "approved": evaluation["approved"],
            "issues": evaluation["issues"],
            "logs": self.logger.get_logs(),
            "metadata": evaluation.get("metadata", {})
        }


def run_agent(user_input: str):
    agent = MainAgent()
    result = agent.handle_message(user_input)
    return result["response"]


In [ ]:
%%writefile project/app.py
try:
    import gradio as gr
except Exception:
    gr = None

from project.main_agent import run_agent


def chat(message, history=None):
    return run_agent(message)


if __name__ == "__main__":
    if gr is None:
        print("Gradio is not installed. Install requirements.txt first.")
    else:
        demo = gr.ChatInterface(
            fn=chat,
            title="Community Health Companion Agent",
            description="Agents for Good demo: safe, non-diagnostic community health guidance."
        )
        demo.launch()


In [ ]:
%%writefile project/run_demo.py
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from project.main_agent import run_agent

if __name__ == "__main__":
    print(run_agent("Hello! This is a demo."))
    print()
    print(run_agent("I have fever and cough for 3 days. What should I do?"))
    print()
    print(run_agent("What are warning signs of dehydration?"))


In [ ]:
%%writefile project/requirements.txt
gradio
python-dotenv
requests


In [ ]:
from project.main_agent import run_agent
print(run_agent("Hello!"))


In [ ]:
!python project/run_demo.py


In [ ]:
!zip -r project.zip project
